# Imports and Setup
The following are imports and setups for the whole file

In [1]:
import boto3
import json
import base64

boto3.setup_default_session(profile_name='personal')

# Create a Bedrock Runtime client in the AWS Region of your choice.
client = boto3.client(service_name='bedrock-runtime', region_name='us-east-1')

In [2]:
# Set the model ID, e.g., Claude 3 Haiku.
model_id = "arn:aws:bedrock:us-east-1:468451602170:inference-profile/us.anthropic.claude-sonnet-4-6"

# Load and Run
Load the prompt, files, and run the AI

In [3]:
# load the system prompt
with open("system-prompt.txt", "r") as file:
    system_prompt = file.read()

files = []

# add the files
with open("input/83d33706-d392-4881-a88c-a92537757d0b.html", "r", encoding='utf8') as file:
    files.append(file.read())

with open("input/6489deab-2b9c-464b-8014-ccb87a9d91d8.html", "r", encoding='utf8') as file:
    files.append(file.read())

with open("schema.json", "r", encoding='utf8') as file:
    schema = json.loads(file.read())

This will run the AI and output the results in a real-time stream

In [4]:
# Setup the request payload
native_request = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 1024,
    "system": system_prompt,
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": files[0]
                },
                {
                    "type": "text",
                    "text": files[1]
                }
            ]
        }
    ],
    "output_config": {
        "format": {
            "type": "json_schema",
            "schema": schema
        }
    }
}

# Convert the request to JSON.
request = json.dumps(native_request)

# Invoke the model with the request.
streaming_response = client.invoke_model_with_response_stream(
    modelId=model_id, body=request
)

# Extract and print the response text in real-time.
for event in streaming_response["body"]:
    chunk = json.loads(event["chunk"]["bytes"])
    if chunk["type"] == "content_block_delta":
        print(chunk["delta"].get("text", ""), end="")

{"changes":true,"differences":["The second version no longer includes the 'Principal Engineer' job listing that was present in the first version."]}